In [1]:
import pandas as pd
from collections import defaultdict
import hashlib

# Build feature set

* Đọc conditions.csv → add DX:<condition_code>

* Đọc procedures.csv → add PROC:<procedure_code>

* Đọc observations.csv → add OBS:<observation_code>

## File conditions

In [6]:
conditions_path = r"D:\Document\Four-year\Second semeter\Mining Massive Data Sets\data\conditions.csv"
conditions = pd.read_csv(conditions_path)
conditions.columns

Index(['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION'], dtype='object')

In [15]:
patient_features = defaultdict(set)

for _, row in conditions.iterrows():
    patient_id = row["PATIENT"]
    condition_code = row["CODE"]

    feature = f"DX:{condition_code}"
    patient_features[patient_id].add(feature)


In [16]:
len(patient_features)

1147

## File procedures

In [17]:
procedures_path = r"D:\Document\Four-year\Second semeter\Mining Massive Data Sets\data\procedures.csv"
procedures = pd.read_csv(procedures_path)
procedures.columns

Index(['START', 'STOP', 'PATIENT', 'ENCOUNTER', 'CODE', 'DESCRIPTION',
       'BASE_COST', 'REASONCODE', 'REASONDESCRIPTION'],
      dtype='object')

In [18]:
for _, row in procedures.iterrows():
    patient_id = row["PATIENT"]
    proc_code = row["CODE"]
    patient_features[patient_id].add(f"PROC:{proc_code}")

In [19]:
len(patient_features)

1162

In [20]:
some_patient = next(iter(patient_features))
some_patient, list(patient_features[some_patient])[:10]

('c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8',
 ['PROC:274474001',
  'DX:16114001',
  'PROC:715252007',
  'PROC:23426006',
  'PROC:428211000124100',
  'PROC:710841007',
  'PROC:171207006',
  'PROC:384700001',
  'PROC:386516004',
  'DX:444814009'])

## File observations

In [21]:
observations_path = r"D:\Document\Four-year\Second semeter\Mining Massive Data Sets\data\observations.csv"
observations = pd.read_csv(observations_path)
observations.columns

Index(['DATE', 'PATIENT', 'ENCOUNTER', 'CATEGORY', 'CODE', 'DESCRIPTION',
       'VALUE', 'UNITS', 'TYPE'],
      dtype='object')

In [22]:
for _, row in observations.iterrows():
    patient_id = row["PATIENT"]
    obs_code = row["CODE"]
    patient_features[patient_id].add(f"OBS:{obs_code}")

In [23]:
len(patient_features)

1163

In [24]:
some_patient = next(iter(patient_features))
features = patient_features[some_patient]

has_dx = any(f.startswith("DX:") for f in features)
has_proc = any(f.startswith("PROC:") for f in features)
has_obs = any(f.startswith("OBS:") for f in features)

some_patient, has_dx, has_proc, has_obs, len(features)

('c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8', True, True, True, 44)

# Build MinHash signatures

- Chọn k = 100 hash

- Với mỗi patient:
    - Với mỗi hash i:
        - min_i = min(hash_i(feature) for feature in set)
    - signature = [min_1, ..., min_100]

In [30]:
def stable_int(s: str) -> int:
    digest = hashlib.sha256(s.encode("utf-8")).digest()
    return int.from_bytes(digest[:8], byteorder="big")

In [31]:
stable_int("DX:123"), stable_int("DX:123"), stable_int("PROC:123")

(6981138120376681323, 6981138120376681323, 15829115508559508635)

In [32]:
import random

random.seed(42)

k = 100
# p là một số nguyên tố lớn (ta dùng 2**61 - 1 là prime phổ biến, đủ lớn để mod)
p= 2**61 - 1

a_list = [random.randrange(1,p) for _ in range(k)]
b_list = [random.randrange(1,p) for _ in range(k)]

a_list[:5], b_list[:5]

([256711980439665054,
  1709899044117486290,
  564682175042572888,
  321752549611034335,
  236337776990707882],
 [1568626733491448721,
  264138331430926778,
  2040183268950970904,
  1731406331274257205,
  1772268004723042249])

In [33]:
def minhash_signature(feature_set):
    sig = []
    for i in range(k):
        min_val = None
        for feat in feature_set:
            x = stable_int(feat)
            h = (a_list[i] * x + b_list[i]) % p
            # cập nhật min_val
            if (min_val is None) or (h < min_val):
                min_val = h
        sig.append(min_val)
    return sig

In [34]:
pid = next(iter(patient_features))
sig = minhash_signature(patient_features[pid])

len(sig), sig[:5]

(100,
 [39960851930717810,
  125391123442168326,
  36740512231783976,
  72335825400209397,
  195291108203425671])

In [35]:
pids = list(patient_features.keys())
p1, p2 = pids[0], pids[1]

#jaccard thật
set1 = patient_features[p1]
set2 = patient_features[p2]

jaccard = len(set1 & set2) / len(set1 | set2)

#minhash similarity
sig1 = minhash_signature(set1)
sig2 = minhash_signature(set2)

minhash_sim = sum(1 for i in range(k) if sig[i] == sig2[i]) / k

jaccard, minhash_sim

(0.40229885057471265, 0.36)

# LSH indexing

- b=20 bands, r=5 rows/band

- band_hash = hash(tuple(signature[j:j+r]))

- bucket[(band_id, band_hash)] add patient_id

In [36]:
b = 20
r = 5

buckets = defaultdict(list)

# lưu signatures
signatures = {}

for pid, feats in patient_features.items():
    sig = minhash_signature(feats)
    signatures[pid] = sig

    # chia band
    for band_id in range(b):
        start = band_id * r
        band = tuple(sig[start:start+r]) # 5 số

        # hash band thành bucket
        band_key = (band_id, hash(band))
        buckets[band_key].append(pid)

len(buckets)

4707

In [40]:
list(patient_features.items())[0]

('c1f1fcaa-82fd-d5b7-3544-c8f9708b06a8',
 {'DX:10509002',
  'DX:16114001',
  'DX:283371005',
  'DX:44465007',
  'DX:444814009',
  'OBS:21000-5',
  'OBS:29463-7',
  'OBS:32207-3',
  'OBS:32623-1',
  'OBS:39156-5',
  'OBS:4544-3',
  'OBS:59576-9',
  'OBS:6690-2',
  'OBS:70274-6',
  'OBS:718-7',
  'OBS:72166-2',
  'OBS:72514-3',
  'OBS:777-3',
  'OBS:785-6',
  'OBS:786-4',
  'OBS:787-2',
  'OBS:789-8',
  'OBS:8302-2',
  'OBS:8462-4',
  'OBS:8480-6',
  'OBS:8867-4',
  'OBS:89204-2',
  'OBS:9279-1',
  'OBS:DALY',
  'OBS:QALY',
  'OBS:QOLS',
  'PROC:171207006',
  'PROC:19490002',
  'PROC:23426006',
  'PROC:274474001',
  'PROC:288086009',
  'PROC:384700001',
  'PROC:386516004',
  'PROC:428211000124100',
  'PROC:430193006',
  'PROC:710841007',
  'PROC:715252007',
  'PROC:76601001',
  'PROC:868187001'})

# Query patient mới

- Build set → signature → bands

- Lấy union các bucket trùng → candidates

- Tính Jaccard thật trên candidates

- Sort desc → Top-K

In [41]:
def get_candidates(pid):
    sig = signatures[pid]
    candidates = set()

    for band_id in range(b):
        start = band_id * r
        band = tuple(sig[start:start+r])
        band_key = (band_id, hash(band))

        for other_pid in buckets[band_key]:
            if other_pid != pid:
                candidates.add(other_pid)
    
    return candidates

cands = get_candidates(p1)
len(cands)

377

# Tính Jaccard thật trên candidates rồi lấy Top-K.

In [42]:
def jaccard_sim(a, b):
    return len(a & b) / len(a | b)

def top_k_similar(pid, K=10):
    base = patient_features[pid]
    cands = get_candidates(pid)

    scored = []
    for other in cands:
        s = jaccard_sim(base, patient_features[other])
        scored.append((other, s))

    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:K]

top10 = top_k_similar(p1, K=10)
top10

[('69bb7f3e-31ab-fee6-9ab6-4a83c24df020', 0.7954545454545454),
 ('d47017f1-c334-f720-16cb-c1be6338b2d3', 0.7954545454545454),
 ('ef0b300f-2534-043f-86f7-f34119bf5bd4', 0.782608695652174),
 ('dc6c06d0-a7d8-100f-c08b-46c93700c188', 0.7659574468085106),
 ('7249d5ba-664d-786e-f1c8-924588615e7a', 0.7608695652173914),
 ('2783a318-7e6d-e3a7-fce7-2a70e66c6ba9', 0.76),
 ('a4ce9a28-c676-33d4-1656-0cb7e4419883', 0.75),
 ('426cade9-82be-f335-7340-c4413e4bdd6a', 0.7450980392156863),
 ('27050e3a-dba0-61b9-3041-373c323e16a4', 0.7391304347826086),
 ('6ca8bd16-892f-5fe6-ce6e-c9e16d6523e9', 0.7291666666666666)]